# Ретроспективная проверка фенологической модели по СЭТ

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vkonov2/AgroPhenology/blob/main/notebooks/pest_phenology_validation.ipynb)

Цель — проверить на исторических данных, насколько дата достижения порога суммы эффективных температур совпадает с датой **той же канонической фенологической стадии**. По умолчанию: яблонная плодожорка, массовое отрождение гусениц, база 10 °C, порог 230 °C·day.

**Научные ограничения.** ERA5-Land — реанализ, а не измерение непосредственно на поле; температура модельной ячейки может отличаться от микроклимата сада. Порог СЭТ показывает вероятное фенологическое окно, не плотность популяции и не необходимость применения пестицида. Для решений нужны полевые осмотры, ловушки и оценка численности. Дата наблюдения может быть датой первого обнаружения при очередном осмотре. Начало накопления — отдельная документируемая гипотеза. Выводы нельзя переносить на другие регионы, поколения и виды без отдельной проверки.

По умолчанию notebook выполняет **только синтетическую техническую демонстрацию** без научного вывода.

In [ ]:
# Bootstrap: локально используется текущий checkout; в Colab репозиторий клонируется.
from pathlib import Path
import importlib.util, os, subprocess, sys

IN_COLAB = 'google.colab' in sys.modules
REPOSITORY_URL = 'https://github.com/vkonov2/AgroPhenology.git'

if IN_COLAB:
    repo_root = Path('/content/AgroPhenology')
    if not (repo_root / 'pyproject.toml').exists():
        try:
            subprocess.check_call(['git', 'clone', '--depth', '1', REPOSITORY_URL, str(repo_root)])
        except subprocess.CalledProcessError as exc:
            raise RuntimeError('Не удалось клонировать репозиторий. Для закрытого репозитория загрузите его архив в Colab или Google Drive и укажите repo_root вручную; не вставляйте токен в notebook.') from exc
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    repo_root = next((p for p in candidates if (p / 'pyproject.toml').exists()), None)
    if repo_root is None:
        raise RuntimeError('Не найден pyproject.toml. Запустите notebook из checkout репозитория.')

if importlib.util.find_spec('agro_phenology') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo_root)])
os.chdir(repo_root)
print(f'Режим: {"Google Colab" if IN_COLAB else "локальный Jupyter"}; репозиторий: {repo_root}')

## Конфигурация

`accumulation_start_date` не выбирается молча. Приоритет имеет дата в строке. `MANUAL_GLOBAL_START_DATE` применяется только к строкам без неё и сохраняется как `manual_global_date`. Если обе даты отсутствуют, расчёт для строки не выполняется. Подбор начала на этих же данных не является независимой валидацией.

In [ ]:
from datetime import datetime, timezone
import math, shutil
import numpy as np
import pandas as pd
from IPython.display import display

from agro_phenology.config import ExperimentConfig, CODLING_MOTH_MODELS
from agro_phenology.observations import load_observations, apply_column_mapping, validate_observations
from agro_phenology.open_meteo import OpenMeteoClient, WeatherResponse
from agro_phenology.validation import validate_records, calculate_summary_metrics, metrics_by_region, methodology_payload, save_results
from agro_phenology.plotting import plot_observation, plot_sample_results

INPUT_MODE = 'template'  # template | local | upload | drive
LOCAL_INPUT_PATH = repo_root / 'data/examples/observations_template.csv'
DRIVE_INPUT_PATH = '/content/drive/MyDrive/observations.csv'
MANUAL_GLOBAL_START_DATE = '2023-04-15'  # демонстрационная гипотеза; None запрещает подстановку
USE_LIVE_OPEN_METEO = False  # True для реальных исторических данных ERA5-Land

COLUMN_MAPPING = {
    'observation_id': 'observation_id', 'latitude': 'latitude', 'longitude': 'longitude',
    'observation_date': 'observation_date', 'pest_name': 'pest_name',
    'observed_stage': 'observed_stage', 'accumulation_start_date': 'accumulation_start_date',
    'crop_name': 'crop_name', 'region': 'region', 'source': 'source',
}

config = ExperimentConfig(
    model=CODLING_MOTH_MODELS['mass_larval_hatch'],
    daily_temperature_methods=('hourly_mean', 'min_max_mean'),
    minimum_valid_hours=20, allow_incomplete_days=False,
    cache_dir=repo_root / 'data/cache/open_meteo', results_dir=repo_root / 'results',
)
print('Режим погоды:', 'живой ERA5-Land' if USE_LIVE_OPEN_METEO else 'СИНТЕТИЧЕСКАЯ ДЕМОНСТРАЦИЯ')

In [ ]:
def select_input_path():
    if INPUT_MODE in {'template', 'local'}:
        return Path(LOCAL_INPUT_PATH)
    if INPUT_MODE == 'upload':
        if not IN_COLAB:
            raise RuntimeError('upload доступен в Google Colab; локально используйте INPUT_MODE=local')
        from google.colab import files
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError('Загрузите ровно один CSV или XLSX файл')
        return Path('/content') / next(iter(uploaded))
    if INPUT_MODE == 'drive':
        if IN_COLAB:
            from google.colab import drive
            drive.mount('/content/drive')
        return Path(DRIVE_INPUT_PATH)
    raise ValueError(f'Неизвестный INPUT_MODE: {INPUT_MODE}')

input_path = select_input_path()
raw_observations = load_observations(input_path)
observations = apply_column_mapping(raw_observations, COLUMN_MAPPING)
observation_report = validate_observations(
    observations, config.model.target_stage_code, MANUAL_GLOBAL_START_DATE,
    target_pest_name=config.model.pest_name,
)
print(f'Файл: {input_path}')
display(observation_report.summary)
display(observation_report.accepted[['observation_id', 'observed_stage', 'accumulation_start_date', 'accumulation_start_rule', 'duplicate_record']])
if MANUAL_GLOBAL_START_DATE is not None:
    print('ВНИМАНИЕ: ручная общая дата начала — демонстрационная гипотеза, не научно подтверждённый параметр.')
if INPUT_MODE == 'template':
    print('ВНИМАНИЕ: строки SYNTH-* синтетические и не являются данными Россельхозмониторинга.')

## Погодные данные и расчёт

Для реального режима используется только Open-Meteo Historical Weather API с `models=era5_land`, `hourly=temperature_2m`, `timezone=auto`, `cell_selection=land`. Автономный режим ниже создаёт искусственный сезонный ряд исключительно для технической проверки pipeline. Дополнительные погодные переменные могут быть сохранены клиентом, но в формулу СЭТ не входят.

In [ ]:
def synthetic_hourly_loader(latitude, longitude, start_date, end_date, variables):
    timestamps = pd.date_range(pd.Timestamp(start_date), pd.Timestamp(end_date) + pd.Timedelta(hours=23), freq='h')
    doy = timestamps.dayofyear.to_numpy()
    hour = timestamps.hour.to_numpy()
    seasonal_mean = 14.0 + 5.0 * np.sin((doy - 100) * np.pi / 100.0)
    temperature = seasonal_mean + 3.0 * np.sin((hour - 8) * 2.0 * np.pi / 24.0)
    hourly = pd.DataFrame({'time': timestamps.strftime('%Y-%m-%dT%H:%M'), 'temperature_2m': temperature})
    metadata = {
        'requested_latitude': latitude, 'requested_longitude': longitude,
        'returned_latitude': latitude, 'returned_longitude': longitude,
        'elevation': None, 'timezone': 'Europe/Moscow', 'timezone_abbreviation': 'MSK',
        'utc_offset_seconds': 10800, 'model': 'synthetic_demo_not_era5_land',
    }
    return WeatherResponse(hourly, metadata, True, Path('synthetic_demo.json'))

client = OpenMeteoClient(config.cache_dir)
hourly_loader = client.fetch_hourly if USE_LIVE_OPEN_METEO else synthetic_hourly_loader
validation_records, daily_tables = validate_records(observation_report.accepted, config, hourly_loader)
summary_metrics = calculate_summary_metrics(validation_records)
regional_metrics = metrics_by_region(validation_records)

excluded_records = pd.concat([observation_report.excluded, observation_report.errors], ignore_index=True)
excluded_records['final_reason'] = excluded_records['exclusion_reason'].where(
    excluded_records['exclusion_reason'].ne(''), excluded_records['error_reason']
)
methodology = methodology_payload(config, datetime.now(timezone.utc).isoformat())
methodology['execution_mode'] = 'live_era5_land' if USE_LIVE_OPEN_METEO else 'synthetic_technical_demo'
if not USE_LIVE_OPEN_METEO:
    methodology['weather_provider'] = 'synthetic generator for technical verification'
    methodology['weather_dataset'] = 'synthetic_demo_not_era5_land'
save_results(validation_records, summary_metrics, excluded_records, methodology, config.results_dir)
display(validation_records)
display(summary_metrics)
if not regional_metrics.empty:
    print('Метрики по регионам с достаточным размером группы:')
    display(regional_metrics)

In [ ]:
# Подробный график одного расчёта. Методы не смешиваются.
ok_rows = validation_records[validation_records['calculation_status'].eq('ok')]
if not ok_rows.empty:
    example = ok_rows.iloc[0]
    key = f"{example['observation_id']}__{example['daily_temperature_method']}"
    fig = plot_observation(
        daily_tables[key], config.model.degree_day_threshold, example['predicted_date'],
        example['observed_date'], example['accumulation_start_date'],
        config.results_dir / 'plots' / 'example_observation.png',
    )
    display(fig)

In [ ]:
plot_paths = plot_sample_results(validation_records, config.results_dir / 'plots')
print('Сохранённые графики:', [p.name for p in plot_paths])
print('Исключения по причинам:')
display(excluded_records.groupby(['record_status', 'final_reason']).size().rename('record_count').reset_index())
print('Метрики являются предварительной оценкой на пилотной синтетической выборке.' if INPUT_MODE == 'template' else 'Интерпретируйте метрики с учётом размера и качества выборки.')

In [ ]:
# Упаковка результатов и скачивание в Colab.
archive_path = Path(shutil.make_archive(str(repo_root / 'phenology_results'), 'zip', config.results_dir))
print(f'ZIP с результатами: {archive_path}')
if IN_COLAB:
    from google.colab import files
    files.download(str(archive_path))

## Вывод

Техническая демонстрация подтверждает только работоспособность расчётного pipeline; она не подтверждает и не опровергает научную гипотезу. После загрузки реальных согласованных наблюдений расчёт показывает ожидаемое окно наступления фенологической стадии и может использоваться для планирования мониторинга.

Для настоящей проверки нужно подтвердить происхождение и точность координат, смысл даты наблюдения, словарь стадий, правило начала накопления, поколение вредителя, регион и методику полевого осмотра.